In [8]:
import sys
import numpy as np
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [9]:
from src.data_loader import load_raw_data
from src.preprocessing import clean_raw_data, chronological_split, fit_scaler, apply_scaler
from src.feature_engineering import build_features
from src.config import MODEL_FEATURE_COLUMNS, TARGET_COL

df_raw = load_raw_data()
df_clean = clean_raw_data(df_raw)
df_features = build_features(df_clean)

print("Shape after feature engineering:", df_features.shape)
print("\nColumns:")
print(df_features.columns.tolist())
df_features.head(3)

2026-09-09 21:51:28 | INFO     | src.data_loader | Loading raw dataset from C:\Users\Ram\OneDrive\Desktop\energy-demand-lstm\data\raw\continuous_dataset.csv
2026-09-09 21:51:30 | INFO     | src.data_loader | Loaded raw dataset: 48048 rows, 17 columns
2026-09-09 21:51:31 | INFO     | src.preprocessing | Cleaning complete: 48048 rows, 4 flagged as diagnostic extreme events (|z| > 4.0)
2026-09-09 21:51:31 | INFO     | src.feature_engineering | Added national weather averages: temp_avg, humidity_avg, precip_avg, wind_avg
2026-09-09 21:51:31 | INFO     | src.feature_engineering | Added calendar features and cyclical encodings
2026-09-09 21:51:31 | INFO     | src.feature_engineering | Added lag features: ['lag_1', 'lag_24', 'lag_48', 'lag_168']
2026-09-09 21:51:31 | INFO     | src.feature_engineering | Added rolling features: rolling_mean_24, rolling_std_24, rolling_mean_168
2026-09-09 21:51:31 | INFO     | src.feature_engineering | Feature engineering complete: 48048 rows before, 47880 rows

,nat_demand,T2M_toc,QV2M_toc,TQL_toc,W2M_toc,T2M_san,QV2M_san,TQL_san,W2M_san,T2M_dav,...,dow_cos,month_sin,month_cos,lag_1,lag_24,lag_48,lag_168,rolling_mean_24,rolling_std_24,rolling_mean_168
datetime,,,,,,,,,,,,,,,,,,,,,
2015-01-10 01:00:00,906.9580,24.976495,0.017215,0.025253,21.335333,23.554620,0.016421,0.051376,12.485627,22.288995,...,-0.222521,0.5,0.866025,949.5031,943.6081,962.4068,970.3450,999.013846,78.862833,1092.047710
2015-01-10 02:00:00,863.5135,24.906274,0.017337,0.034378,22.177057,23.429712,0.016337,0.038712,12.949576,22.296899,...,-0.222521,0.5,0.866025,906.9580,917.0640,928.1036,912.1755,997.486758,80.323661,1091.670407
2015-01-10 03:00:00,848.4447,24.879724,0.017512,0.045349,22.742188,23.309412,0.016292,0.028526,13.091533,22.325037,...,-0.222521,0.5,0.866025,863.5135,895.9092,917.9997,900.2688,995.255488,83.341887,1091.380752


In [10]:
splits = chronological_split(df_features)

scale_columns = MODEL_FEATURE_COLUMNS + [TARGET_COL]
scaler = fit_scaler(splits.train, scale_columns)

train_scaled = apply_scaler(splits.train, scaler, scale_columns)
val_scaled = apply_scaler(splits.val, scaler, scale_columns)
test_scaled = apply_scaler(splits.test, scaler, scale_columns)

print("Train scaled shape:", train_scaled.shape)
print(train_scaled[scale_columns].describe().loc[["min", "max"]])

2026-09-09 21:51:43 | INFO     | src.preprocessing | Chronological split: train=33516 (2015-01-10 01:00:00 to 2018-11-06 12:00:00), val=7182 (2018-11-06 13:00:00 to 2019-09-01 18:00:00), test=7182 (2019-09-01 19:00:00 to 2020-06-27 00:00:00)
2026-09-09 21:51:43 | INFO     | src.preprocessing | Scaler fit on training data: 21 columns, 33516 rows
Train scaled shape: (33516, 41)
     temp_avg  humidity_avg  precip_avg  wind_avg  hour_sin  hour_cos  \
min       0.0           0.0         0.0       0.0       0.0       0.0   
max       1.0           1.0         1.0       1.0       1.0       1.0   

     dow_sin  dow_cos  month_sin  month_cos  ...  holiday  school  lag_1  \
min      0.0      0.0        0.0        0.0  ...      0.0     0.0    0.0   
max      1.0      1.0        1.0        1.0  ...      1.0     1.0    1.0   

     lag_24  lag_48  lag_168  rolling_mean_24  rolling_std_24  \
min     0.0     0.0      0.0              0.0             0.0   
max     1.0     1.0      1.0              

In [11]:
from src.sequence_builder import create_sequences_for_all_splits
from src.config import MODEL_FEATURE_COLUMNS, DEFAULT_LOOKBACK, FORECAST_HORIZON

sequences = create_sequences_for_all_splits(
    train_scaled, val_scaled, test_scaled,
    feature_columns=MODEL_FEATURE_COLUMNS,
    lookback=DEFAULT_LOOKBACK,
    horizon=FORECAST_HORIZON,
)

for split_name, (X, y) in sequences.items():
    print(f"{split_name}: X={X.shape}, y={y.shape}")

2026-09-09 21:51:45 | INFO     | src.sequence_builder | Built 33325 sequences: X=(33325, 168, 20), y=(33325, 24) (lookback=168, horizon=24, dropped 191 boundary rows)
2026-09-09 21:51:45 | INFO     | src.sequence_builder | Split 'train': X=(33325, 168, 20), y=(33325, 24)
2026-09-09 21:51:46 | INFO     | src.sequence_builder | Built 6991 sequences: X=(6991, 168, 20), y=(6991, 24) (lookback=168, horizon=24, dropped 191 boundary rows)
2026-09-09 21:51:46 | INFO     | src.sequence_builder | Split 'val': X=(6991, 168, 20), y=(6991, 24)


2026-09-09 21:51:46 | INFO     | src.sequence_builder | Built 6991 sequences: X=(6991, 168, 20), y=(6991, 24) (lookback=168, horizon=24, dropped 191 boundary rows)
2026-09-09 21:51:46 | INFO     | src.sequence_builder | Split 'test': X=(6991, 168, 20), y=(6991, 24)
train: X=(33325, 168, 20), y=(33325, 24)
val: X=(6991, 168, 20), y=(6991, 24)
test: X=(6991, 168, 20), y=(6991, 24)


In [12]:
X_train, y_train = sequences["train"]

# Manually verify sequence 0: its y should equal rows [lookback : lookback+horizon]
# of the target column in train_scaled.
expected_y = train_scaled[TARGET_COL].values[DEFAULT_LOOKBACK : DEFAULT_LOOKBACK + FORECAST_HORIZON]
print("Matches expected y:", np.allclose(y_train[0], expected_y))

# Confirm shapes align with configured lookback/horizon
print("Lookback dimension correct:", X_train.shape[1] == DEFAULT_LOOKBACK)
print("Num features correct:", X_train.shape[2] == len(MODEL_FEATURE_COLUMNS))
print("Horizon dimension correct:", y_train.shape[1] == FORECAST_HORIZON)

Matches expected y: True
Lookback dimension correct: True
Num features correct: True
Horizon dimension correct: True
